# Local text classification workbook

Edit the settings in the next cell, then run the cells from top to bottom. The workbook calls `app.py`, so the notebook and command-line workflow produce the same output format.

In [31]:
# User settings

INPUT = "data/input"                   # JSON file or folder of JSON files
OUTPUT_DIR = "data/output"             # Folder for classified JSON output
MODEL = "qwen3:0.6b"                  # Ollama model tag
BASE_URL = "http://localhost:11434"    # Ollama base URL

QUESTION = "Does the following abstract make a policy claim? Answer yes or no."

ID_FIELD = "scopus_id"                 # Stable record ID field
TEXT_FIELD = "abstract"                # Field sent to the LLM for review
CONTEXT_FIELDS = ["title"]             # Extra fields sent as context; use [] for none

CONCURRENCY = 4                        # Try 1, 2, 4 depending on your machine
LIMIT =  10                           # Use an integer like 10 for testing; None for no limit
FORCE = True                           # True reclassifies existing output, False skips already classified files


In [32]:
# Optional: show installed Ollama models

import subprocess

result = subprocess.run(["ollama", "list"], text=True, capture_output=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("Could not run `ollama list`. Make sure Ollama is installed and running.")
    print(result.stderr)


NAME                                                     ID              SIZE      MODIFIED      
hf.co/unsloth/Qwen3-4B-Instruct-2507-GGUF:Q4_K_M         5611a5f21404    2.5 GB    9 months ago     
gpt-oss:20b                                              aa4295ac10c3    13 GB     9 months ago     
gemma3:270m                                              e7d36fb2c3b3    291 MB    9 months ago     
gemma3:12b                                               f4031aab637d    8.1 GB    9 months ago     
qwen3:8b                                                 500a1f067a9f    5.2 GB    9 months ago     
hf.co/unsloth/Qwen3-30B-A3B-Instruct-2507-GGUF:Q4_K_M    a40c031fcbbd    18 GB     10 months ago    
gemma3:4b                                                a2af6cc3eb7f    3.3 GB    10 months ago    
gemma3n:e4b                                              15cb39fd9394    7.5 GB    10 months ago    
qwen3:0.6b                                               7df6b6e09427    522 MB    10 months a

In [33]:
# Preview the input files and first record

import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()

def display_path(path):
    path = Path(path)
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT.resolve()))
    except (OSError, RuntimeError, ValueError):
        pass
    if path.is_absolute():
        return str(Path(path.parent.name) / path.name) if path.parent.name else path.name
    return str(path)

def discover_files(input_value):
    input_path = Path(input_value)
    if input_path.is_file():
        return [input_path]
    return sorted(input_path.glob("*.json"))

files = discover_files(INPUT)
print(f"Found {len(files)} JSON file(s):")
for path in files:
    print(f"- {display_path(path)}")

if files:
    with files[0].open("r", encoding="utf-8") as file:
        data = json.load(file)
    if isinstance(data, dict):
        for key in ("records", "items", "abstracts", "data"):
            if isinstance(data.get(key), list):
                data = data[key]
                break
    print(f"\nFirst file record count: {len(data)}")
    if data:
        first = data[0]
        print(f"\nID field ({ID_FIELD}):")
        print(first.get(ID_FIELD, ""))
        for field in CONTEXT_FIELDS:
            print(f"\nContext field ({field}):")
            print(first.get(field, ""))
        print(f"\nText field sent to the LLM ({TEXT_FIELD}) preview:")
        print((first.get(TEXT_FIELD, "") or "")[:700])


Found 1 JSON file(s):
- data/input/lancet_public_health_S2764808104.json

First file record count: 455

ID field (scopus_id):
SCOPUS_ID:85019951336

Context field (title):
Total and cause-specific mortality before and after the onset of the Greek economic crisis: an interrupted time-series analysis

Text field sent to the LLM (abstract) preview:
Background Greece was one of the countries hit the hardest by the 2008 financial crisis in Europe. Yet, evidence on the effect of the crisis on total and cause-specific mortality remains unclear. We explored whether the economic crisis affected the trend of overall and cause-specific mortality rates. Methods We used regional panel data from the Hellenic Statistical Authority to assess mortality trends by age, sex, region, and cause in Greece between January, 2001, and December, 2013. We used Eurostat data to calculate monthly age-standardised mortality rates per 100 000 inhabitants for each region. Data were divided into two subperiods: before 

In [34]:
# Run classification

import re
import shlex
import sys
import time

def format_duration(seconds):
    minutes, seconds = divmod(float(seconds), 60)
    hours, minutes = divmod(int(minutes), 60)
    if hours:
        return f"{hours}h {minutes}m {seconds:.1f}s"
    if minutes:
        return f"{minutes}m {seconds:.1f}s"
    return f"{seconds:.1f}s"

def estimate_duration(records, records_per_second):
    if not records_per_second:
        return "n/a"
    return format_duration(records / records_per_second)

run_cmd = [
    sys.executable,
    "app.py",
    "--input", str(INPUT),
    "--output-dir", str(OUTPUT_DIR),
    "--model", str(MODEL),
    "--base-url", str(BASE_URL),
    "--question", str(QUESTION),
    "--id-field", str(ID_FIELD),
    "--text-field", str(TEXT_FIELD),
    "--context-fields", ",".join(CONTEXT_FIELDS),
    "--concurrency", str(CONCURRENCY),
]
display_cmd = ["python3", *run_cmd[1:]]

if LIMIT is not None:
    run_cmd.extend(["--limit", str(LIMIT)])
    display_cmd.extend(["--limit", str(LIMIT)])
if FORCE:
    run_cmd.append("--force")
    display_cmd.append("--force")

print(shlex.join(display_cmd))
started = time.perf_counter()
result = subprocess.run(run_cmd, text=True, capture_output=True)
elapsed_seconds = time.perf_counter() - started
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Classification failed")

records_processed = sum(
    int(match.group(1))
    for match in re.finditer(r"\|\s+(\d+)\s+to do", result.stdout)
)
records_per_second = records_processed / elapsed_seconds if records_processed else 0

print("\nTiming")
print(f"Elapsed: {format_duration(elapsed_seconds)}")
print(f"Records processed this run: {records_processed}")
if records_processed:
    print(f"Throughput: {records_per_second:.2f} records/s ({records_per_second * 60:.0f} records/min)")
    print("\nRough time estimates at this speed")
    for target_records in (1_000, 10_000, 40_000):
        print(f"{target_records:,} records: {estimate_duration(target_records, records_per_second)}")
else:
    print("No new records were classified; existing output was reused.")

LAST_RUN = {
    "elapsed_seconds": elapsed_seconds,
    "records_processed": records_processed,
    "records_per_second": records_per_second,
    "command": display_cmd,
}


python3 app.py --input data/input --output-dir data/output --model qwen3:0.6b --base-url http://localhost:11434 --question 'Does the following abstract make a policy claim? Answer yes or no.' --id-field scopus_id --text-field abstract --context-fields title --concurrency 4 --limit 10 --force
Using Ollama model qwen3:0.6b at http://localhost:11434
Using ID field 'scopus_id', text field 'abstract', context fields ['title']
data/input/lancet_public_health_S2764808104.json: 10 total | 0 already done | 10 to do
  10/10 (4.40/s, ETA 0.0 min)
Done. Results in data/output/lancet_public_health_S2764808104_classified.json


Timing
Elapsed: 2.4s
Records processed this run: 10
Throughput: 4.21 records/s (253 records/min)

Rough time estimates at this speed
1,000 records: 3m 57.5s
10,000 records: 39m 34.8s
40,000 records: 2h 38m 19.2s


In [35]:
# Summarise outputs

from collections import Counter

output_files = sorted(Path(OUTPUT_DIR).glob("*_classified.json"))
print(f"Found {len(output_files)} classified output file(s):")

for output_file in output_files:
    with output_file.open("r", encoding="utf-8") as file:
        records = json.load(file)
    counts = Counter(
        (record.get("classification") or {}).get("answer", "missing")
        for record in records
    )
    print(f"\n{display_path(output_file)}")
    print(f"Records: {len(records)}")
    for answer, count in sorted(counts.items()):
        print(f"{answer}: {count}")


Found 1 classified output file(s):

data/output/lancet_public_health_S2764808104_classified.json
Records: 10
unknown: 10
